# Recommender systems

## Libraries and constants

In [80]:
import gc
import os

from pathlib import Path

import numpy as np
import pandas as pd
from utils import get_null_info

from surprise import Dataset, Reader, SVD, accuracy, prediction_algorithms
from surprise.model_selection import train_test_split as surprise_train_test_split
from surprise.model_selection import GridSearchCV as surprise_GridSearchCV

In [54]:
# --- CONSTANTS ---
RANDOM_SEED = 42
n_jobs = os.cpu_count()-1

## Data import

In [22]:
links_raw = pd.read_csv(Path('data') / 'links.csv')
links_raw

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0
...,...,...,...
9737,193581,5476944,432131.0
9738,193583,5914996,445030.0
9739,193585,6397426,479308.0
9740,193587,8391976,483455.0


The mapping of movie id to imdb and tmdb, I don't see any use in this dataset.

In [23]:
del links_raw
gc.collect()

7

In [24]:
movies_raw = pd.read_csv(Path('data') / 'movies.csv')
movies_raw

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
9739,193585,Flint (2017),Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation


Maps a movie id to its title and genres, can be useful to map the final predictions.

In [25]:
ratings_raw = pd.read_csv(Path('data') / 'ratings.csv')
ratings_raw

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


The main dataframe with movies, users and ratings

In [26]:
tags_raw = pd.read_csv(Path('data') / 'tags.csv')
tags_raw

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200
...,...,...,...,...
3678,606,7382,for katie,1171234019
3679,606,7936,austere,1173392334
3680,610,3265,gun fu,1493843984
3681,610,3265,heroic bloodshed,1493843978


Interesting moments in every movie with timestamps, we don't need it.

In [27]:
del tags_raw
gc.collect()

14

## EDA

In [28]:
ratings_raw.describe()

,userId,movieId,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


I don't see why we'd need the timestamp

In [29]:
ratings_raw = ratings_raw.drop(columns=['timestamp'])

In [30]:
duplicates_rating = ratings_raw[ratings_raw.duplicated()]
duplicates_rating

,userId,movieId,rating


In [31]:
ratings_raw.dtypes

userId       int64
movieId      int64
rating     float64
dtype: object

In [32]:
get_null_info(ratings_raw)

No missing values are found in the data_frame


""


In [9]:
ratings_raw['rating'].value_counts()

rating
4.0    26818
3.0    20047
5.0    13211
3.5    13136
4.5     8551
2.0     7551
2.5     5550
1.0     2811
1.5     1791
0.5     1370
Name: count, dtype: int64

## Train (surprise library) - Task 1

### Prepare the data, train the baseline (SVD)

In [39]:
# Tell the surprise library the rating scale
scale = (ratings_raw.rating.min(), ratings_raw.rating.max())
reader = Reader(rating_scale=scale)

# Hand over the 𝒦: ONLY the three columns, in the order (user, item, rating).
data = Dataset.load_from_df(ratings_raw[['userId', 'movieId', 'rating']], reader)

# Split into train and test
df_train, df_test = surprise_train_test_split(data, test_size=0.2, random_state=RANDOM_SEED, shuffle=True)

# The stochastic gradient-based SVD
algo = SVD(
    n_factors=100,              # k  — how many components to create to explain a user and an item entity
    n_epochs=20,                # how many times to loop through train
    lr_all=0.005,               # α  — learning rate to update all parameters
    reg_all=0.02,               # λ  — regularization term for all parameter updates
    biased=True,                # do use biases
    random_state=RANDOM_SEED,   # fixed random seed for reproducibility
)

# Train
algo.fit(df_train)

In [81]:
def evaluate(model: prediction_algorithms, df_train: pd.DataFrame, df_test: pd.DataFrame):
    # Evaluate train
    train_tuples = df_train.build_testset()         # because surprise parses the train dataset separately for efficiency, we have to convert it back
    train_predictions = model.test(train_tuples)
    rmse_train = accuracy.rmse(train_predictions, verbose=False)

    # Evaluate test
    predictions = model.test(df_test)
    rmse_test = accuracy.rmse(predictions, verbose=False)

    print(f"RMSE (train) : {rmse_train:.4f}")
    print(f"RMSE (test) : {rmse_test:.4f}")

In [ ]:
evaluate(algo, df_train, df_test)

RMSE (train) : 0.6354
RMSE (test) : 0.8807


Great generalizability, but since train still have room before it overfits, we may push it further.

### Tuning with GridSearch and cross-validation (SVD)

In [ ]:
# Grid to try
param_grid = {
    'n_factors': [50, 100],      # k — number of latent factors
    'n_epochs':  [20, 30],       # passes over the data
    'lr_all':    [0.005, 0.01],  # α — learning rate
    'reg_all':   [0.02, 0.1],    # λ — regularization
}

gs = surprise_GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=5, n_jobs=n_jobs)
gs.fit(data)          # using full data, not train, CV makes its own holdouts

print(gs.best_score['rmse'])    # the best RMSE it found
print(gs.best_params['rmse'])   # the winning combination

0.8556163530245955
{'n_factors': 100, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}


In [ ]:
# Grid to try
param_grid = {
    'n_factors': [150, 200],        # k — number of latent factors
    'n_epochs':  [40, 50],          # passes over the data
    'lr_all':    [0.05, 0.1],       # α — learning rate
    'reg_all':   [0.3, 0.5],        # λ — regularization
}

gs = surprise_GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=5, n_jobs=n_jobs)
gs.fit(data)          # using full data, not train, CV makes its own holdouts for each epoch

print(gs.best_score['rmse'])    # the best RMSE it found
print(gs.best_params['rmse'])   # the winning combination

0.8869037047592807
{'n_factors': 200, 'n_epochs': 40, 'lr_all': 0.05, 'reg_all': 0.3}


In [ ]:
# 1. Grab the winning settings found by your grid search
best_settings = gs.best_params['rmse']

# 2. Instantiate a fresh model using the winning parameters
final_algo = SVD(
    n_factors=best_settings['n_factors'],
    n_epochs=best_settings['n_epochs'],
    lr_all=best_settings['lr_all'],
    reg_all=best_settings['reg_all'],
    biased=True,
    random_state=RANDOM_SEED,
)

# 3. Build a trainset out of 100% of your raw data (no more splitting!)
full_data = data.build_full_trainset()

# 4. Train the final model on everything
final_algo.fit(full_data)

In [ ]:
evaluate(final_algo, df_train, df_test)

RMSE (train) : 0.8350
RMSE (test) : 0.8389


In [46]:
# 1. Grab the winning settings found by your grid search
best_settings = gs.best_params['rmse']

# 2. Instantiate a fresh model using the winning parameters
final_algo = SVD(
    n_factors=best_settings['n_factors'],
    n_epochs=best_settings['n_epochs'],
    lr_all=best_settings['lr_all'],
    reg_all=best_settings['reg_all'],
    biased=True,
    random_state=RANDOM_SEED,
)

# 3. Build a trainset out of 100% of your raw data (no more splitting!)
full_data = data.build_full_trainset()

# 4. Train the final model on everything
final_algo.fit(full_data)

In [ ]:
evaluate(final_algo, df_train, df_test)

RMSE (train) : 0.8350
RMSE (test) : 0.8389


Closest gap, right before overfitting, we're keeping it.

### Trying out SVDpp, NME

In [64]:
from surprise import SVD, SVDpp, NMF
from surprise.model_selection import GridSearchCV as surprise_GridSearchCV

# 1. Створюємо окремі сітки параметрів для кожного алгоритму
# Примітка: Для SVD++ сітка менша, оскільки він рахується значно довше.
grids = {
    'SVD++': {
        'algo_class': SVDpp,
        'param_grid': {
            'n_factors': [20, 30],
            'n_epochs': [10],
            'lr_all':    [0.005],
            'reg_all':   [0.02],
        },
    },
    'NMF': {
        'algo_class': NMF,
        'param_grid': {
            'n_factors': [15, 30],       # NMF зазвичай потребує менше факторів
            'n_epochs': [15],
            # В NMF регуляризація розділена для користувачів та предметів
            'reg_pu':    [0.06],
            'reg_qi':    [0.06],
        },
    },
}

best_models = {}

# 2. Запускаємо пошук по сітці для кожного алгоритму
for name, config in grids.items():
    print(f"--- Running GridSearchCV for {name} ---")

    gs = surprise_GridSearchCV(
        config['algo_class'],
        config['param_grid'],
        measures=['rmse'],
        cv=5,
        n_jobs=n_jobs, # Використовує всі ядра процесора для прискорення
    )
    gs.fit(data)

    # Зберігаємо результати
    best_models[name] = {
        'score': gs.best_score['rmse'],
        'params': gs.best_params['rmse'],
    }

    print(f"Best RMSE for {name}: {gs.best_score['rmse']}")
    print(f"Best Params for {name}: {gs.best_params['rmse']}\n")

# 3. Визначаємо абсолютного переможця
winner = min(best_models, key=lambda k: best_models[k]['score'])

print("========================================")
print(f"🥇 ОПТИМАЛЬНИЙ АЛГОРИТМ: {winner}")
print(f"Кращий результат RMSE: {best_models[winner]['score']}")
print(f"Параметри для фінального навчання: {best_models[winner]['params']}")
print("========================================")


--- Running GridSearchCV for SVD++ ---
Best RMSE for SVD++: 0.872830462665964
Best Params for SVD++: {'n_factors': 30, 'n_epochs': 10, 'lr_all': 0.005, 'reg_all': 0.02}

--- Running GridSearchCV for NMF ---
Best RMSE for NMF: 0.9609876715559841
Best Params for NMF: {'n_factors': 15, 'n_epochs': 15, 'reg_pu': 0.06, 'reg_qi': 0.06}

🥇 ОПТИМАЛЬНИЙ АЛГОРИТМ: SVD++
Кращий результат RMSE: 0.872830462665964
Параметри для фінального навчання: {'n_factors': 30, 'n_epochs': 10, 'lr_all': 0.005, 'reg_all': 0.02}


In [65]:
best_models[winner]

{'score': 0.872830462665964,
 'params': {'n_factors': 30, 'n_epochs': 10, 'lr_all': 0.005, 'reg_all': 0.02}}

In [66]:
# 1. Grab the winning settings found by your grid search
best_settings = gs.best_params['rmse']

# 2. Instantiate a fresh model using the winning parameters
svdpp_algo = SVD(
    n_factors=best_models[winner]['params']['n_factors'],
    n_epochs=best_models[winner]['params']['n_epochs'],
    lr_all=best_models[winner]['params']['lr_all'],
    reg_all=best_models[winner]['params']['reg_all'],
    biased=True,
    random_state=RANDOM_SEED,
)

# 3. Build a trainset out of 100% of your raw data (no more splitting!)
full_data = data.build_full_trainset()

# 4. Train the final model on everything
svdpp_algo.fit(full_data)

In [ ]:
evaluate(svdpp_algo, df_train, df_test)

RMSE (train) : 0.8306
RMSE (test) : 0.8381


tuning svdpp

In [68]:
# Grid to try
param_grid = {
    'n_factors': [40, 50],        # k — number of latent factors
    'n_epochs':  [5, 15],          # passes over the data
    'lr_all':    [0.005],       # α — learning rate
    'reg_all':   [0.02],        # λ — regularization
}

gs = surprise_GridSearchCV(SVDpp, param_grid, measures=['rmse', 'mae'], cv=5, n_jobs=n_jobs)
gs.fit(data)          # using full data, not train, CV makes its own holdouts for each epoch

print(gs.best_score['rmse'])    # the best RMSE it found
print(gs.best_params['rmse'])   # the winning combination

0.8652416644230716
{'n_factors': 50, 'n_epochs': 15, 'lr_all': 0.005, 'reg_all': 0.02}


In [72]:
# 1. Grab the winning settings found by your grid search
best_settings = gs.best_params['rmse']

# 2. Instantiate a fresh model using the winning parameters
final_algo = SVDpp(
    n_factors=best_settings['n_factors'],
    n_epochs=best_settings['n_epochs'],
    lr_all=best_settings['lr_all'],
    reg_all=best_settings['reg_all'],
    random_state=RANDOM_SEED,
)

# 3. Build a trainset out of 100% of your raw data (no more splitting!)
full_data = data.build_full_trainset()

# 4. Train the final model on everything
final_algo.fit(full_data)

In [74]:
evaluate(final_algo, df_train, df_test)

RMSE (train) : 0.7388
RMSE (test) : 0.7455


In [75]:
final_algo.predict(uid=1, iid=1).est      # user 1's predicted rating for movie 1

4.696244384344909

Actual recommendations — the point of the whole exercise. Predict every movie this user hasn't rated, sort, take the top 10, and decode to titles via movies.csv:

In [76]:
user_id = 1

rated   = ratings_raw.loc[ratings_raw.userId == user_id, 'movieId']
unrated = movies_raw.loc[~movies_raw.movieId.isin(rated), 'movieId']

preds = [(mid, final_algo.predict(user_id, mid).est) for mid in unrated]
preds.sort(key=lambda x: x[1], reverse=True)

for mid, est in preds[:10]:
    title = movies_raw.loc[movies_raw.movieId == mid, 'title'].values[0]
    print(f"{est:.2f}  {title}")

5.00  Shawshank Redemption, The (1994)
5.00  Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)
5.00  Godfather, The (1972)
5.00  Rear Window (1954)
5.00  Casablanca (1942)
5.00  Streetcar Named Desire, A (1951)
5.00  Good, the Bad and the Ugly, The (Buono, il brutto, il cattivo, Il) (1966)
5.00  Godfather: Part II, The (1974)
5.00  Rosemary's Baby (1968)
5.00  Lord of the Rings: The Fellowship of the Ring, The (2001)


### Analytical implementation (PCA-style SVD, not trainable)

In [ ]:
import numpy as np
import pandas as pd


def analytical_SVD(M_df: pd.DataFrame):
    """SVD from scratch via the eigenvalue route:  M = U Σ Vᵀ (thin)."""
    # NOTE (correction): sparse rating tables have NaN holes. Analytical SVD needs
    # EVERY entry, so we're forced to fillna(0). This is the "poisonous lie" from the
    # notes — a hole becomes a 0 rating. Fine for a genuinely complete matrix
    # (images, covariance); wrong for recommenders. The mock at the bottom shows why.
    M_numeric = M_df.fillna(0).to_numpy()
    m, n = M_numeric.shape

    # ---------- Step 1: Make it square so we can find eigenvectors ----------
    # MMᵀ (m×m) shares eigenvectors with U;  MᵀM (n×n) shares them with V.
    # Pick whichever is smaller to keep the hand/symbolic work light.
    is_wide = m < n
    if is_wide:
        square_M = M_numeric @ M_numeric.T   # m×m  -> gives U first
    else:
        square_M = M_numeric.T @ M_numeric   # n×n  -> gives V first

    # ---------- Step 2 + 4: Eigenvalues AND eigenvectors via np.linalg.eigh ----------
    # NOTE (major correction): the original did det()→sp.solve()→sp.nullspace(). That
    # chain BREAKS on 3×3+ matrices: sp.solve returns roots with imaginary float noise,
    # and once λ is a float, (S - λI) is no longer EXACTLY singular, so sp.nullspace()
    # returns empty and the eigenvectors vanish. Symbolic nullspace needs exact
    # rank-deficiency; floats destroy it.
    #
    # eigh is the right tool: it is built for REAL SYMMETRIC matrices (which MMᵀ and
    # MᵀM always are), returns real eigenvalues + orthonormal eigenvectors in one shot,
    # and correctly handles repeated eigenvalues (the Gram-Schmidt caveat, for free).
    eigenvalues, eigenvectors = np.linalg.eigh(square_M)

    # eigh returns them SMALLEST-first; reverse to SVD convention σ₁ ≥ σ₂ ≥ …
    order = np.argsort(eigenvalues)[::-1]
    M_eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    # ---------- Step 3: Singular values σ = √λ ----------
    # Clamp tiny negative float noise (e.g. -1e-16) to 0 before sqrt, else nan.
    M_singular_values = np.sqrt(np.maximum(0.0, M_eigenvalues))

    # The eigenvectors of the square matrix ARE the "first" family (already unit-length):
    # is_wide  -> columns of U ;  not wide -> columns of V
    first_matrix = eigenvectors

    # ---------- Step 5: Recover the mirror family by going back through M ----------
    # is_wide  -> have U, get V:  vᵢ = (1/σ) Mᵀ uᵢ
    # not wide -> have V, get U:  uᵢ = (1/σ) M  vᵢ
    second_space_vectors = []
    for idx, sigma in enumerate(M_singular_values):
        # NOTE (correction, REJECTED Gemini's fix): a σ=0 direction was crushed to
        # nothing — it has no valid mirror vector. Gemini appended np.zeros(...) as a
        # "filler", but a zero column is NOT unit-length or orthogonal and it silently
        # breaks U @ Σ @ Vᵀ. Correct behavior is to skip it -> thin SVD.
        if sigma < 1e-9:
            continue

        if is_wide:
            u_i = first_matrix[:, idx]
            v_i = (1.0 / sigma) * (M_numeric.T @ u_i)
            second_space_vectors.append(v_i)
        else:
            v_i = first_matrix[:, idx]
            u_i = (1.0 / sigma) * (M_numeric @ v_i)
            second_space_vectors.append(u_i)

    second_matrix = np.column_stack(second_space_vectors)

    # ---------- Step 6: Map both families back to standard U, Σ, Vᵀ ----------
    if is_wide:
        U = first_matrix          # eigenvectors of MMᵀ
        V_T = second_matrix.T     # recovered
    else:
        U = second_matrix         # recovered
        V_T = first_matrix.T      # eigenvectors of MᵀM

    # Keep only the non-zero singular values (thin Σ), so shapes match the dropped σ=0.
    nonzero_sigmas = M_singular_values[M_singular_values > 1e-9]
    Sigma = np.diag(nonzero_sigmas)

    return U, Sigma, V_T

In [ ]:
# =====================================================================
# --- VERIFICATION WITH A REALISTIC MOCK MOVIE TABLE (step by step) ---
# =====================================================================
# 4 Users (rows), 3 Movies (cols). NaN = unwatched "holes".
mock_rating_data = {
    "SciFi_Movie":   [5.0, 4.0, np.nan, 1.0],
    "Action_Movie":  [4.0, np.nan, 2.0, 1.0],
    "Romance_Movie": [np.nan, 1.0, 4.0, 5.0],
}
R_table = pd.DataFrame(mock_rating_data, index=["User_1", "User_2", "User_3", "User_4"])

print("--- Step A: Original sparse table (NaN = hole) ---")
print(R_table)

print("\n--- Step B: What the function actually decomposes (fillna(0)) ---")
# This is the lie in numbers: 'unwatched' is now the rating 0.0, LOWER than the
# real worst rating (1.0). So every hole becomes 'hated more than anything'.
print(R_table.fillna(0))

print("\n--- Step C: Run analytical SVD on the zero-filled table ---")
U, Sigma, V_T = analytical_SVD(R_table)
print("U shape:", U.shape, "| Sigma shape:", Sigma.shape, "| Vᵀ shape:", V_T.shape)
print("Singular values (diagonal of Σ):", np.round(np.diag(Sigma), 3))

print("\n--- Step D: Reconstruct U @ Σ @ Vᵀ ---")
R_reconstructed = U @ Sigma @ V_T
recon_df = pd.DataFrame(np.round(R_reconstructed, 2),
                        index=R_table.index, columns=R_table.columns)
print(recon_df)

print("\n--- Step E: The point — look ONLY at the hole cells ---")
# The holes were: (User_3, SciFi), (User_2, Action), (User_1, Romance).
# The math reconstructs the *zero-filled* matrix faithfully, so these come back
# NEAR 0 — not a sensible 3–5 prediction. Analytical SVD 'predicts' unwatched
# movies as ~0 = "recommend nothing". That is exactly why recommenders need the
# GD version, which never lets a hole enter the math.
for user, movie in [("User_3", "SciFi_Movie"),
                    ("User_2", "Action_Movie"),
                    ("User_1", "Romance_Movie")]:
    print(f"  hole ({user}, {movie}) -> reconstructed {recon_df.loc[user, movie]:>5}   (a real prediction should be ~3–5)")

If even doing this task, I wouldn't consider implementing some pseudo bias-free version as the task is trying to impose. I'd rather create a real thing.

In [19]:
# Step 1: obtain the sparse pivot grid full of 0s (your initial data where for each user there are values in columns for certain movies and empty cells on the ones they didn't rate)
# Step 2: make a dataframe that only includes existing data (e.g., user_id, movie_id, rating)
# Step 3: make a holdout test
# Step 4: Train

def GD_SVD(
        u_idx: pd.Series, i_idx: pd.Series,                     # user index, item index (manual positional mappings to handle random ids from disrupting indexing)
        ratings: pd.Series, num_users: int, num_items: int,     # train ratings, unique users, unique items
        latent_factors: int = 100,                              # how many components to create to explain a user and an item entity
        learning_rate: float = 0.02,                            # learning rate to update all parameters
        regularization: float = 0.02,                           # regularization term for all parameter updates
        epochs: int = 20,                                       # how many times to loop through train
        random_seed: int = 42,                                  # fixed random seed for reproducibility
        ):

    # ---------- STEP 1: Init the params, convert and rename for convenience ----------
    rng = np.random.default_rng(random_seed)
    P = rng.normal(0, 0.1, (num_users, latent_factors)) # user (P)references
    Q = rng.normal(0, 0.1, (num_items, latent_factors)) # item (Q)ualities
    b_u, b_i = np.zeros(num_users), np.zeros(num_items) # bias user, bias item
    mu = ratings.mean()

    u_idx, i_idx, ratings = np.asarray(u_idx), np.asarray(i_idx), np.asarray(ratings)
    n = len(ratings)

    # ---------- STEP 2: Training loop ----------
    # for each epoch
    for ep in range(epochs):
        # shuffle the train each epoch
        for k in rng.permutation(n):
            u, i, r = u_idx[k], i_idx[k], ratings[k]                    # user_id pos index, item_id pos index, true rating of user_id for item_id

            # copy to preserve the original value (because it takes part in computing both P and Q)
            p_u = P[u].copy()
            q_i = Q[i].copy()

            # forward pass
            r_hat = mu + b_u[u] + b_i[i] + p_u @ q_i

            # error
            e = r - r_hat

            # update the user, the item, the bias
            P[u]  += learning_rate * (e * q_i - regularization * p_u)
            Q[i]  += learning_rate * (e * p_u - regularization * q_i)
            b_u[u] += learning_rate * (e - regularization * b_u[u])
            b_i[i] += learning_rate * (e - regularization * b_i[i])

    return mu, b_u, b_i, P, Q

def predict(df_test: pd.DataFrame, mu: float, b_u: float, b_i: float, P: np.ndarray, Q: np.ndarray):
    u, i = df_test['u'].to_numpy(), df_test['i'].to_numpy()
    dot = np.sum(P[u] * Q[i], axis=1)      # element row-wise p_u · q_i, then summing across columns
    return mu + b_u[u] + b_i[i] + dot

In [ ]:
# unique user ids, unique item ids
user_ids, item_ids = ratings_raw['userId'].unique(), ratings_raw['movieId'].unique()

# number of unique users, number of unique items
num_users, num_items = len(user_ids), len(item_ids)

# positional mapping to account for random df indexing {df_index : ordinal index}
user_to_idx = {raw: pos for pos, raw in enumerate(user_ids)}
item_to_idx = {raw: pos for pos, raw in enumerate(item_ids)}

# line up the raw created indexes on a copy
ratings = ratings_raw.copy()
ratings['u'] = ratings['userId'].map(user_to_idx)
ratings['i'] = ratings['movieId'].map(item_to_idx)

In [ ]:
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

# data preparation: 80% train, 20% test
df_train, df_test = train_test_split(ratings, test_size=0.2, random_state=RANDOM_SEED)

# train
mu, b_u, b_i, P, Q = GD_SVD(df_train['u'], df_train['i'], df_train['rating'],
                            num_users, num_items, latent_factors=100, learning_rate=0.005, epochs=20)

In [ ]:
# evaluate
r_true_train = df_train['rating']
r_hat_train = predict(df_train, mu, b_u, b_i, P, Q)
rmse_train = root_mean_squared_error(r_true_train, r_hat_train)

r_true_test = df_test['rating']
r_hat_test = predict(df_test, mu, b_u, b_i, P, Q)
rmse_test = root_mean_squared_error(r_true_test, r_hat_test)

print(f"RMSE (train) : {rmse_train:.4f}")
print(f"RMSE (test) : {rmse_test:.4f}")

RMSE (train) : 0.6395
RMSE (test) : 0.8847
